In [1]:
# %% CELL 0 -- mount the offline package bundle + face-alignment hub cache
# (same bundle used by the training notebook -- face-alignment still needs
# to run here, since landmark detection is part of the real inference path)
import os, shutil, sys, zipfile

OFFLINE_PACKAGES_SRC = "/kaggle/input/datasets/ruwadnaswan/package-df-msib/offline_packages"      # EDIT
FACE_ALIGNMENT_HUB_SRC = "/kaggle/input/datasets/ruwadnaswan/package-df-msib/face_alignment_hub"   # EDIT

def _materialize(source, dest_dir):
    if os.path.isdir(source):
        return source
    if os.path.isfile(source) and source.lower().endswith(".zip"):
        if not (os.path.isdir(dest_dir) and os.listdir(dest_dir)):
            os.makedirs(dest_dir, exist_ok=True)
            with zipfile.ZipFile(source, "r") as z:
                z.extractall(dest_dir)
        return dest_dir
    raise FileNotFoundError(f"{source} is neither an existing directory nor a .zip file")

SITE_PACKAGES_DIR = _materialize(OFFLINE_PACKAGES_SRC, "/kaggle/working/offline_site_packages")
sys.path.insert(0, SITE_PACKAGES_DIR)

HUB_DIR = "/kaggle/working/hub"
if not (os.path.isdir(HUB_DIR) and os.listdir(HUB_DIR)):
    if os.path.isdir(FACE_ALIGNMENT_HUB_SRC):
        shutil.copytree(FACE_ALIGNMENT_HUB_SRC, HUB_DIR, dirs_exist_ok=True)
    else:
        _materialize(FACE_ALIGNMENT_HUB_SRC, HUB_DIR)

import torch  # noqa: E402
torch.hub.set_dir(HUB_DIR)
import face_alignment  # noqa: E402
print("face_alignment OK, version:", getattr(face_alignment, "__version__", "unknown"))
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "bf16 supported:", torch.cuda.is_bf16_supported())


# %% CELL 1 -- imports & config (inference-only subset of the training CFG --
# same field names/values so a checkpoint trained with the training notebook's
# defaults loads with matching shapes)
import math, time, statistics
import numpy as np
import cv2
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
from typing import List, Tuple

torch.set_float32_matmul_precision("high")
NEG_INF = -1e4


class CFG:
    img_size = 224
    patch_size = 16
    num_frames = 8
    num_landmarks = 68
    max_hop = 2
    num_freq_bands = 6

    vis_dim = 384
    vis_depth = 12
    vis_heads = 6
    graph_dim = 192
    graph_depth = 12
    graph_heads = 6
    mlp_ratio = 4.0
    num_classes = 2

    guide_bias_clip = 8.0

    amp = True
    amp_dtype = "bf16"

    celebdf_crop_margin = 0.4   # used for the video-preprocessing face crop

    device = "cuda" if torch.cuda.is_available() else "cpu"


def resolve_amp_dtype(cfg: "CFG"):
    if cfg.amp_dtype == "bf16" and torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if cfg.amp_dtype == "bf16":
        print("[amp] bf16 requested but unsupported on this GPU -- falling back to fp16")
    return torch.float16


# %% CELL 2 -- landmark graph utilities (identical to the training notebook's CELL 3)
def _ibug68_edges() -> List[Tuple[int, int]]:
    edges = []
    edges += [(i, i + 1) for i in range(0, 16)]
    edges += [(i, i + 1) for i in range(17, 21)]
    edges += [(i, i + 1) for i in range(22, 26)]
    edges += [(i, i + 1) for i in range(27, 30)]
    edges += [(i, i + 1) for i in range(31, 35)]
    edges += [(27, 31), (30, 33)]
    edges += [(i, i + 1) for i in range(36, 41)] + [(41, 36)]
    edges += [(i, i + 1) for i in range(42, 47)] + [(47, 42)]
    edges += [(i, i + 1) for i in range(48, 59)] + [(59, 48)]
    edges += [(i, i + 1) for i in range(60, 67)] + [(67, 60)]
    edges += [(19, 37), (24, 44), (33, 51)]
    return edges


EDGES_68 = _ibug68_edges()
NUM_LANDMARKS = 68


def build_adjacency(num_nodes: int = NUM_LANDMARKS, edges=EDGES_68) -> np.ndarray:
    adj = np.zeros((num_nodes, num_nodes), dtype=bool)
    for i, j in edges:
        adj[i, j] = True
        adj[j, i] = True
    return adj


def build_hyper_adjacency(num_landmarks: int = NUM_LANDMARKS) -> np.ndarray:
    base = build_adjacency(num_landmarks)
    adj = np.zeros((num_landmarks + 1, num_landmarks + 1), dtype=bool)
    adj[1:, 1:] = base
    adj[0, 1:] = True
    adj[1:, 0] = True
    return adj


def all_pairs_hops(adj: np.ndarray, max_hop: int) -> np.ndarray:
    n = adj.shape[0]
    hop = np.full((n, n), -1, dtype=np.int64)
    for src in range(n):
        hop[src, src] = 0
        q = deque([src]); visited = {src}
        while q:
            u = q.popleft()
            if hop[src, u] >= max_hop:
                continue
            for v in np.nonzero(adj[u])[0]:
                if v not in visited:
                    visited.add(v)
                    hop[src, v] = hop[src, u] + 1
                    q.append(v)
    return hop


def node_degree(adj: np.ndarray) -> np.ndarray:
    return adj.sum(axis=1).astype(np.int64)


# %% CELL 3 -- shared embedding modules (identical to training notebook's CELL 7)
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=384):
        super().__init__()
        self.grid = img_size // patch_size
        self.num_patches = self.grid * self.grid
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.reshape(B * T, C, H, W)
        x = self.proj(x).flatten(2).transpose(1, 2)
        D = x.shape[-1]
        return x.reshape(B, T, -1, D)


def sine_cosine_pe(coords, num_freq_bands):
    freqs = (2.0 ** torch.arange(num_freq_bands, device=coords.device, dtype=coords.dtype)) * math.pi
    args = coords.unsqueeze(-1) * freqs
    enc = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
    return enc.flatten(-2)


class NodeEmbed(nn.Module):
    def __init__(self, embed_dim, num_freq_bands, num_nodes=NUM_LANDMARKS, max_degree=8):
        super().__init__()
        in_dim = 3 * 2 * num_freq_bands
        self.proj = nn.Linear(in_dim, embed_dim)
        degree = torch.from_numpy(node_degree(build_adjacency(num_nodes))).long()
        self.register_buffer("degree", degree)
        self.degree_embed = nn.Embedding(max_degree + 1, embed_dim)
        self.num_freq_bands = num_freq_bands

    def forward(self, landmarks_norm):
        pe = sine_cosine_pe(landmarks_norm, self.num_freq_bands)
        h = self.proj(pe)
        deg = self.degree.clamp(max=self.degree_embed.num_embeddings - 1)
        return h + self.degree_embed(deg)[None, None, :, :]


# %% CELL 4 -- spatial attention (vision stream) -- identical to training notebook's CELL 8
class RelPosBias(nn.Module):
    def __init__(self, grid, num_heads):
        super().__init__()
        self.grid = grid
        num_rel = (2 * grid - 1) * (2 * grid - 1)
        self.table = nn.Parameter(torch.zeros(num_rel, num_heads))
        coords = torch.stack(torch.meshgrid(torch.arange(grid), torch.arange(grid), indexing="ij"), dim=-1).reshape(-1, 2)
        rel = coords[:, None, :] - coords[None, :, :] + (grid - 1)
        idx = rel[..., 0] * (2 * grid - 1) + rel[..., 1]
        self.register_buffer("index", idx)
        nn.init.trunc_normal_(self.table, std=0.02)

    def forward(self, has_cls: bool):
        L = self.grid * self.grid
        bias = self.table[self.index.reshape(-1)].reshape(L, L, -1).permute(2, 0, 1)
        if has_cls:
            H = bias.shape[0]
            full = bias.new_zeros(H, L + 1, L + 1)
            full[:, 1:, 1:] = bias
            bias = full
        return bias


class SpatialAttention(nn.Module):
    def __init__(self, dim, num_heads, grid, has_cls=True):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.rel_pos = RelPosBias(grid, num_heads)
        self.has_cls = has_cls

    def forward(self, x):
        B, T, L, D = x.shape
        qkv = self.qkv(x).reshape(B, T, L, 3, self.num_heads, self.head_dim).permute(3, 0, 1, 4, 2, 5)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn + self.rel_pos(self.has_cls)[None, None]
        attn = attn.softmax(dim=-1)
        out = attn @ v
        out = out.transpose(2, 3).reshape(B, T, L, D)
        return self.proj(out), attn


# %% CELL 5 -- topology-aware spatial attention (graph stream) -- identical to CELL 9
class TopoSpatialAttention(nn.Module):
    def __init__(self, dim, num_heads, num_landmarks=NUM_LANDMARKS, max_hop=2):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        adj = build_hyper_adjacency(num_landmarks)
        hop = all_pairs_hops(adj, max_hop)
        self.register_buffer("hop", torch.from_numpy(hop).long())
        self.hop_embed = nn.Parameter(torch.zeros(max_hop + 1, num_heads))
        nn.init.trunc_normal_(self.hop_embed, std=0.02)

    def _bias(self, device):
        N = self.hop.shape[0]
        valid = self.hop >= 0
        idx = self.hop.clamp(min=0)
        gathered = self.hop_embed[idx.reshape(-1)].reshape(N, N, self.num_heads)
        bias = torch.where(valid.unsqueeze(-1), gathered, torch.full_like(gathered, NEG_INF))
        return bias.permute(2, 0, 1)

    def forward(self, x):
        B, T, N, D = x.shape
        qkv = self.qkv(x).reshape(B, T, N, 3, self.num_heads, self.head_dim).permute(3, 0, 1, 4, 2, 5)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn + self._bias(x.device)[None, None]
        attn = attn.softmax(dim=-1)
        out = attn @ v
        out = out.transpose(2, 3).reshape(B, T, N, D)
        return self.proj(out), attn


# %% CELL 6 -- Kronecker Temporal Attention (KTA) -- identical to CELL 10
import functools

@functools.lru_cache(maxsize=8)
def kronecker_temporal_mask(T, L, device, dtype):
    I_t = torch.eye(T, device=device)
    J_m = torch.ones(L, L, device=device)
    I_m = torch.eye(L, device=device)
    M = torch.kron(I_t, (J_m - I_m))
    return torch.where(M > 0, torch.full_like(M, NEG_INF), torch.zeros_like(M)).to(dtype)


class KroneckerTemporalAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, guide_bias=None):
        B, T, L, D = x.shape
        x_flat = x.reshape(B, T * L, D)
        qkv = self.qkv(x_flat).reshape(B, T * L, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        raw_logits = (q @ k.transpose(-2, -1)) * self.scale

        mask = kronecker_temporal_mask(T, L, x.device, raw_logits.dtype)[None, None]
        logits = raw_logits + mask
        if guide_bias is not None:
            logits = logits + guide_bias[:, None]

        attn = logits.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T * L, D)
        out = self.proj(out).reshape(B, T, L, D)
        return out, raw_logits


def extract_landmark_block(raw_logits, T, N):
    B, H = raw_logits.shape[0], raw_logits.shape[1]
    Np1 = N + 1
    r = raw_logits.view(B, H, T, Np1, T, Np1)
    r = r[:, :, :, 1:, :, 1:]
    return r.reshape(B, H, T * N, T * N)


def scatter_graph_bias_to_vision(graph_raw_logits_land, patch_idx, T, N, L, bias_clip=8.0):
    B = graph_raw_logits_land.shape[0]
    device = graph_raw_logits_land.device
    out_dtype = graph_raw_logits_land.dtype
    graph_bias = graph_raw_logits_land.float().mean(dim=1)

    TL, TN = T * L, T * N
    t_idx = torch.arange(T, device=device).repeat_interleave(N)
    node_to_patch = patch_idx.reshape(B, TN)
    row_full = t_idx.unsqueeze(0) * L + node_to_patch

    idx_i = row_full.unsqueeze(2).expand(B, TN, TN).reshape(B, -1)
    idx_j = row_full.unsqueeze(1).expand(B, TN, TN).reshape(B, -1)
    batch_offset = (torch.arange(B, device=device) * TL * TL).unsqueeze(1)
    flat_idx = (idx_i * TL + idx_j + batch_offset).reshape(-1)

    values = graph_bias.reshape(-1)
    out_sum = torch.zeros(B * TL * TL, device=device, dtype=torch.float32)
    out_cnt = torch.zeros(B * TL * TL, device=device, dtype=torch.float32)
    out_sum.scatter_add_(0, flat_idx, values)
    out_cnt.scatter_add_(0, flat_idx, torch.ones_like(values))

    out = (out_sum / out_cnt.clamp(min=1.0)).reshape(B, TL, TL)
    out = out.clamp(min=-bias_clip, max=bias_clip)
    return out.to(out_dtype)


# %% CELL 7 -- spatiotemporal blocks -- identical to CELL 11
class Mlp(nn.Module):
    def __init__(self, dim, ratio=4.0, drop=0.0):
        super().__init__()
        hidden = int(dim * ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.fc2(self.drop(self.act(self.fc1(x))))


class VisionSTB(nn.Module):
    def __init__(self, dim, num_heads, grid, mlp_ratio=4.0, has_cls=True):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.spatial = SpatialAttention(dim, num_heads, grid, has_cls=has_cls)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp1 = Mlp(dim, mlp_ratio)
        self.norm3 = nn.LayerNorm(dim)
        self.temporal = KroneckerTemporalAttention(dim, num_heads)
        self.norm4 = nn.LayerNorm(dim)
        self.mlp2 = Mlp(dim, mlp_ratio)

    def forward(self, x, guide_bias=None):
        sa_out, _ = self.spatial(self.norm1(x))
        x = x + sa_out
        x = x + self.mlp1(self.norm2(x))
        ta_out, raw_logits = self.temporal(self.norm3(x), guide_bias=guide_bias)
        x = x + ta_out
        x = x + self.mlp2(self.norm4(x))
        return x, raw_logits


class GraphSTB(nn.Module):
    def __init__(self, dim, num_heads, num_landmarks, max_hop, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.spatial = TopoSpatialAttention(dim, num_heads, num_landmarks, max_hop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp1 = Mlp(dim, mlp_ratio)
        self.norm3 = nn.LayerNorm(dim)
        self.temporal = KroneckerTemporalAttention(dim, num_heads)
        self.norm4 = nn.LayerNorm(dim)
        self.mlp2 = Mlp(dim, mlp_ratio)

    def forward(self, x):
        sa_out, _ = self.spatial(self.norm1(x))
        x = x + sa_out
        x = x + self.mlp1(self.norm2(x))
        ta_out, raw_logits = self.temporal(self.norm3(x))
        x = x + ta_out
        x = x + self.mlp2(self.norm4(x))
        return x, raw_logits


# %% CELL 8 -- full G2V2former model -- identical to training notebook's CELL 12
class G2V2former(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.T = cfg.num_frames
        self.N = cfg.num_landmarks
        self.patch_size = cfg.patch_size
        self.img_size = cfg.img_size
        self.guide_bias_clip = cfg.guide_bias_clip

        self.patch_embed = PatchEmbed(cfg.img_size, cfg.patch_size, 3, cfg.vis_dim)
        self.grid = self.patch_embed.grid
        self.L = self.patch_embed.num_patches
        self.vis_pos_embed = nn.Parameter(torch.zeros(1, 1, self.L, cfg.vis_dim))
        self.vis_time_embed = nn.Parameter(torch.zeros(1, cfg.num_frames, 1, cfg.vis_dim))
        self.vis_cls_token = nn.Parameter(torch.zeros(1, 1, 1, cfg.vis_dim))
        self.vis_blocks = nn.ModuleList([
            VisionSTB(cfg.vis_dim, cfg.vis_heads, self.grid, cfg.mlp_ratio, has_cls=True)
            for _ in range(cfg.vis_depth)
        ])
        self.vis_norm = nn.LayerNorm(cfg.vis_dim)

        self.node_embed = NodeEmbed(cfg.graph_dim, cfg.num_freq_bands, cfg.num_landmarks)
        self.hyper_node = nn.Parameter(torch.zeros(1, 1, 1, cfg.graph_dim))
        self.graph_time_embed = nn.Parameter(torch.zeros(1, cfg.num_frames, 1, cfg.graph_dim))
        self.graph_blocks = nn.ModuleList([
            GraphSTB(cfg.graph_dim, cfg.graph_heads, cfg.num_landmarks, cfg.max_hop, cfg.mlp_ratio)
            for _ in range(cfg.graph_depth)
        ])
        self.graph_norm = nn.LayerNorm(cfg.graph_dim)

        self.head = nn.Linear(cfg.vis_dim + cfg.graph_dim, cfg.num_classes)

        nn.init.trunc_normal_(self.vis_pos_embed, std=0.02)
        nn.init.trunc_normal_(self.vis_time_embed, std=0.02)
        nn.init.trunc_normal_(self.vis_cls_token, std=0.02)
        nn.init.trunc_normal_(self.hyper_node, std=0.02)
        nn.init.trunc_normal_(self.graph_time_embed, std=0.02)

    def forward(self, frames, landmarks_norm, landmarks_px):
        B, T = frames.shape[0], frames.shape[1]

        x = self.patch_embed(frames)
        x = x + self.vis_pos_embed + self.vis_time_embed
        cls_tok = self.vis_cls_token.expand(B, T, -1, -1)
        x = torch.cat([cls_tok, x], dim=2)
        L_full = x.shape[2]

        g = self.node_embed(landmarks_norm) + self.graph_time_embed
        hnode = self.hyper_node.expand(B, T, -1, -1)
        g = torch.cat([hnode, g], dim=2)

        with torch.no_grad():
            grid = self.grid
            px = landmarks_px.clamp(min=0, max=self.img_size - 1)
            col = (px[..., 0] // self.patch_size).long().clamp(max=grid - 1)
            row = (px[..., 1] // self.patch_size).long().clamp(max=grid - 1)
            patch_idx = (row * grid + col) + 1

        guide_bias = None
        for vblock, gblock in zip(self.vis_blocks, self.graph_blocks):
            g, g_raw_logits = gblock(g)
            g_land_logits = extract_landmark_block(g_raw_logits, T, self.N)
            guide_bias = scatter_graph_bias_to_vision(
                g_land_logits, patch_idx, T, self.N, L_full, bias_clip=self.guide_bias_clip
            )
            x, _ = vblock(x, guide_bias=guide_bias)

        x = self.vis_norm(x)
        g = self.graph_norm(g)

        vis_cls_seq = x[:, :, 0, :]
        vis_cls_out = vis_cls_seq.mean(dim=1)
        graph_hyper_out = g[:, :, 0, :].mean(dim=1)

        feat = torch.cat([vis_cls_out, graph_hyper_out], dim=-1)
        logits = self.head(feat)
        return logits, vis_cls_out, graph_hyper_out, vis_cls_seq


# %% CELL 9 -- load a checkpoint saved by the training notebook
# The training notebook wrote checkpoints under its own /kaggle/working/ckpts,
# which does NOT persist to a separate notebook/session. Publish that folder
# (or just the .pt files you need) as a Kaggle Dataset and mount it here.
CKPT_INPUT_DIR = "/kaggle/input/notebooks/sleepytmzd/lets-see-2/ckpts"   # EDIT: wherever you published notebook 1's ckpts


def load_model_from_checkpoint(cfg: "CFG", ckpt_path: str) -> G2V2former:
    model = G2V2former(cfg).to(cfg.device)
    state = torch.load(ckpt_path, map_location=cfg.device, weights_only=True)
    model.load_state_dict(state)
    model.eval()
    return model


# %% CELL 10 -- face-crop helper for video frames (identical to training notebook's CELL 6)
def _crop_box_and_landmarks(preds, h, w, margin, img_size):
    if not preds:
        side = int(min(h, w) * 0.8)
        y0, x0 = (h - side) // 2, (w - side) // 2
        x1, y1 = x0 + side, y0 + side
        lmk = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
        return (x0, y0, x1, y1), lmk

    p = preds[0].astype(np.float32)
    xy = p[:, :2]
    x_min, y_min = xy.min(axis=0)
    x_max, y_max = xy.max(axis=0)
    bw, bh = max(x_max - x_min, 1.0), max(y_max - y_min, 1.0)
    cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
    side = max(bw, bh) * (1.0 + margin)
    x0, x1 = int(max(0, cx - side / 2)), int(min(w, cx + side / 2))
    y0, y1 = int(max(0, cy - side / 2)), int(min(h, cy + side / 2))
    if x1 <= x0 or y1 <= y0:
        x0, y0, x1, y1 = 0, 0, w, h

    sx, sy = img_size / (x1 - x0), img_size / (y1 - y0)
    lmk = p.copy()
    lmk[:, 0] = (lmk[:, 0] - x0) * sx
    lmk[:, 1] = (lmk[:, 1] - y0) * sy
    if not np.isfinite(lmk).all():
        lmk = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
    return (x0, y0, x1, y1), lmk


# %% CELL 11 -- preprocessing: raw file on disk -> model-ready tensors (batch size 1)
def _preprocess_still_image(img_bgr, fa_model, cfg: "CFG"):
    """Image pipeline: no face-box crop, direct resize (matches face-cropped
    still datasets like LCC-FASD / Asian-Fakes). A still image is a 'clip of
    length 1' -- replicated across cfg.num_frames temporal slots."""
    h0, w0 = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    try:
        preds = fa_model.get_landmarks(img_rgb)
    except Exception:
        preds = None
    lmk = preds[0].astype(np.float32) if preds else np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
    if not np.isfinite(lmk).all():
        lmk = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)

    img_resized = cv2.resize(img_rgb, (cfg.img_size, cfg.img_size))
    sx, sy = cfg.img_size / w0, cfg.img_size / h0
    lmk = lmk.copy()
    lmk[:, 0] *= sx
    lmk[:, 1] *= sy

    img_t = torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0
    img_t = (img_t - 0.5) / 0.5
    frames = img_t.unsqueeze(0).repeat(cfg.num_frames, 1, 1, 1)

    lmks_px = torch.from_numpy(lmk[:, :2]).unsqueeze(0).repeat(cfg.num_frames, 1, 1)
    lmks_norm = lmks_px.clone()
    lmks_norm[..., 0] /= cfg.img_size
    lmks_norm[..., 1] /= cfg.img_size
    z = torch.from_numpy(lmk[:, 2]).unsqueeze(0).repeat(cfg.num_frames, 1)
    z_range = (z.max() - z.min()) + 1e-6
    z_norm = (z - z.min()) / z_range
    lmks_norm = torch.cat([lmks_norm, z_norm.unsqueeze(-1)], dim=-1)

    return frames.unsqueeze(0), lmks_norm.unsqueeze(0), lmks_px.unsqueeze(0)


def _preprocess_video(video_path, fa_model, cfg: "CFG"):
    """Video pipeline: decode cfg.num_frames sampled frames, face-landmark
    detect + crop + resize each (matches CelebDF-style raw video with
    background)."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        raise RuntimeError(f"Could not read frame count from {video_path}")
    T = cfg.num_frames
    target = set(np.linspace(0, total - 1, min(T, total)).round().astype(int).tolist())

    raw, i = {}, 0
    while len(raw) < len(target):
        ok = cap.grab()
        if not ok:
            break
        if i in target:
            ok, frame = cap.retrieve()
            if ok:
                raw[i] = frame
        i += 1
    cap.release()

    chosen = sorted(raw.keys())
    if not chosen:
        raise RuntimeError(f"No frames decoded from {video_path}")
    if len(chosen) < T:
        chosen = chosen + [chosen[-1]] * (T - len(chosen))
    chosen = chosen[:T]

    imgs, lmks = [], []
    for idx in chosen:
        frame = raw[idx]
        h, w = frame.shape[:2]
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        try:
            preds = fa_model.get_landmarks(frame_rgb)
        except Exception:
            preds = None
        box, lmk = _crop_box_and_landmarks(preds, h, w, cfg.celebdf_crop_margin, cfg.img_size)
        x0, y0, x1, y1 = box
        crop = frame_rgb[y0:y1, x0:x1]
        if crop.size == 0:
            crop = frame_rgb
        crop = cv2.resize(crop, (cfg.img_size, cfg.img_size))
        imgs.append(crop)
        lmks.append(lmk)

    imgs = np.stack(imgs)
    lmks = np.stack(lmks).astype(np.float32)
    imgs_t = torch.from_numpy(imgs).permute(0, 3, 1, 2).float() / 255.0
    imgs_t = (imgs_t - 0.5) / 0.5
    lmks_px = torch.from_numpy(lmks[..., :2])
    lmks_norm = lmks_px.clone()
    lmks_norm[..., 0] /= cfg.img_size
    lmks_norm[..., 1] /= cfg.img_size
    z = torch.from_numpy(lmks[..., 2])
    z_range = (z.max() - z.min()) + 1e-6
    z_norm = (z - z.min()) / z_range
    lmks_norm = torch.cat([lmks_norm, z_norm.unsqueeze(-1)], dim=-1)

    return imgs_t.unsqueeze(0), lmks_norm.unsqueeze(0), lmks_px.unsqueeze(0)


# %% CELL 12 -- timing harness + benchmark functions
def _summarize_latency(records, tag):
    def _stats(key):
        vals = sorted(r[key] for r in records)
        return {
            "mean_ms": statistics.mean(vals),
            "median_ms": statistics.median(vals),
            "std_ms": statistics.pstdev(vals) if len(vals) > 1 else 0.0,
            "min_ms": vals[0], "max_ms": vals[-1],
            "p95_ms": vals[int(0.95 * (len(vals) - 1))],
        }

    summary = {"tag": tag, "n_runs": len(records),
               "preprocess": _stats("preprocess_ms"), "model": _stats("model_ms"), "total": _stats("total_ms")}

    print(f"\n=== Latency: {tag}  (n={len(records)} runs, batch size 1) ===")
    for stage in ("preprocess", "model", "total"):
        s = summary[stage]
        print(f"  {stage:11s} mean={s['mean_ms']:7.2f}ms  median={s['median_ms']:7.2f}ms  "
              f"p95={s['p95_ms']:7.2f}ms  std={s['std_ms']:6.2f}ms  "
              f"min={s['min_ms']:7.2f}ms  max={s['max_ms']:7.2f}ms")
    print(f"  -> throughput: {1000.0 / summary['total']['mean_ms']:.2f} inferences/sec "
          f"(end-to-end, incl. face-landmark detection)")
    return summary


@torch.no_grad()
def _run_forward(model, frames, lmks_norm, lmks_px, cfg, device):
    amp_enabled = cfg.amp and device == "cuda"
    amp_dtype = resolve_amp_dtype(cfg)
    frames = frames.to(device, non_blocking=True)
    lmks_norm = lmks_norm.to(device, non_blocking=True)
    lmks_px = lmks_px.to(device, non_blocking=True)
    if device == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
        logits, *_ = model(frames, lmks_norm, lmks_px)
        probs = F.softmax(logits.float(), dim=-1)[:, 1]
    if device == "cuda":
        torch.cuda.synchronize()
    t2 = time.perf_counter()
    return probs.item(), t1, t2


def benchmark_image_latency(image_path, model, cfg: "CFG", fa_model, n_warmup=5, n_runs=50):
    device = cfg.device
    model.eval()
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(image_path)

    def _run_once():
        t0 = time.perf_counter()
        frames, lmks_norm, lmks_px = _preprocess_still_image(img_bgr, fa_model, cfg)
        p_live, t1, t2 = _run_forward(model, frames, lmks_norm, lmks_px, cfg, device)
        return {"preprocess_ms": (t1 - t0) * 1000, "model_ms": (t2 - t1) * 1000,
                "total_ms": (t2 - t0) * 1000, "p_live": p_live}

    for _ in range(n_warmup):
        _run_once()
    records = [_run_once() for _ in range(n_runs)]
    return _summarize_latency(records, tag=f"IMAGE ({os.path.basename(image_path)})")


def benchmark_video_latency(video_path, model, cfg: "CFG", fa_model, n_warmup=3, n_runs=20):
    device = cfg.device
    model.eval()

    def _run_once():
        t0 = time.perf_counter()
        frames, lmks_norm, lmks_px = _preprocess_video(video_path, fa_model, cfg)
        p_live, t1, t2 = _run_forward(model, frames, lmks_norm, lmks_px, cfg, device)
        return {"preprocess_ms": (t1 - t0) * 1000, "model_ms": (t2 - t1) * 1000,
                "total_ms": (t2 - t0) * 1000, "p_live": p_live}

    for _ in range(n_warmup):
        _run_once()
    records = [_run_once() for _ in range(n_runs)]
    return _summarize_latency(records, tag=f"VIDEO ({os.path.basename(video_path)}, {cfg.num_frames} frames sampled)")


@torch.no_grad()
def benchmark_model_forward_only(model, cfg: "CFG", batch_size=1, n_warmup=10, n_runs=100):
    """Pure model-forward latency on random tensors -- isolates the
    transformer from face-landmark-detection/preprocessing cost."""
    device = cfg.device
    model.eval()
    frames = torch.randn(batch_size, cfg.num_frames, 3, cfg.img_size, cfg.img_size, device=device)
    lmks_norm = torch.rand(batch_size, cfg.num_frames, cfg.num_landmarks, 3, device=device)
    lmks_px = (torch.rand(batch_size, cfg.num_frames, cfg.num_landmarks, 2, device=device) * cfg.img_size)
    amp_enabled = cfg.amp and device == "cuda"
    amp_dtype = resolve_amp_dtype(cfg)

    def _run_once():
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
            model(frames, lmks_norm, lmks_px)
        if device == "cuda":
            torch.cuda.synchronize()
        dt = (time.perf_counter() - t0) * 1000
        return {"preprocess_ms": 0.0, "model_ms": dt, "total_ms": dt}

    for _ in range(n_warmup):
        _run_once()
    records = [_run_once() for _ in range(n_runs)]
    return _summarize_latency(records, tag=f"MODEL-ONLY (batch={batch_size}, random input, no preprocessing)")


# %% CELL 13 -- usage
cfg = CFG()
CKPT_PATH = os.path.join(CKPT_INPUT_DIR, "g2v2former_best.pt")   # EDIT: or "g2v2former_lcc_then_celebdf_best.pt", etc.
model = load_model_from_checkpoint(cfg, CKPT_PATH)

fa_model = face_alignment.FaceAlignment(
    face_alignment.LandmarksType.THREE_D, flip_input=False, device=cfg.device
)

IMAGE_PATH_FOR_LATENCY = "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/spoof/spoof_1000.png"   # EDIT
VIDEO_PATH_FOR_LATENCY = "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-synthesis/id0_id16_0000.mp4"   # EDIT

image_latency = benchmark_image_latency(IMAGE_PATH_FOR_LATENCY, model, cfg, fa_model, n_warmup=2, n_runs=5)
video_latency = benchmark_video_latency(VIDEO_PATH_FOR_LATENCY, model, cfg, fa_model, n_warmup=2, n_runs=5)
model_only_latency = benchmark_model_forward_only(model, cfg, batch_size=1, n_warmup=2, n_runs=5)

face_alignment OK, version: 1.5.0
torch: 2.10.0+cu128 cuda available: True
GPU: Tesla T4 bf16 supported: True


/kaggle/input/datasets/ruwadnaswan/package-df-msib/offline_packages/face_alignment/api.py:130: UserWarning: Compiling face alignment model (one-time cost). Subsequent runs will be faster.
  warnings.warn(
W0901 16:06:59.187000 23 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode



=== Latency: IMAGE (spoof_1000.png)  (n=5 runs, batch size 1) ===
  preprocess  mean=  97.91ms  median=  96.63ms  p95=  96.70ms  std=  4.37ms  min=  94.74ms  max= 106.49ms
  model       mean= 131.49ms  median= 131.41ms  p95= 131.76ms  std=  0.68ms  min= 130.43ms  max= 132.53ms
  total       mean= 229.39ms  median= 227.49ms  p95= 228.11ms  std=  4.23ms  min= 226.51ms  max= 237.79ms
  -> throughput: 4.36 inferences/sec (end-to-end, incl. face-landmark detection)

=== Latency: VIDEO (id0_id16_0000.mp4, 8 frames sampled)  (n=5 runs, batch size 1) ===
  preprocess  mean=1180.89ms  median=1181.49ms  p95=1184.41ms  std=  8.84ms  min=1167.21ms  max=1194.19ms
  model       mean= 134.56ms  median= 134.29ms  p95= 134.91ms  std=  1.31ms  min= 133.13ms  max= 136.88ms
  total       mean=1315.45ms  median=1316.41ms  p95=1318.71ms  std=  8.87ms  min=1300.34ms  max=1327.75ms
  -> throughput: 0.76 inferences/sec (end-to-end, incl. face-landmark detection)

=== Latency: MODEL-ONLY (batch=1, random input